<div dir="rtl" style="text-align: right;">


# چالش آماری علم داده، ترم ۳ (شماره ۱)
---

| مورد | توضیح |
|---|---|
| نام | Kiarash Akbari |
| مجموعه | آزمایشگاه فناوری‌های مالی داتین |
| تاریخ | ۱۵ مهر ۱۴۰۵ (1405-07-15) |
| فایل داده | `bank_midterm.csv` (۴۰۰ مشتری کارت بانکی) |
| متغیر پاسخ | `monthly_spend` (رگرسیون خطی) و `default` (رگرسیون لجستیک) |

---
## مرحله‌ی اول: بررسی و آماده‌سازی داده



### هدف این بخش
قبل از برازش مدل، داده را بررسی می‌کنم و فقط چیزهایی را اصلاح می‌کنم که واقعاً خطا هستند. در پایان هم توضیح می‌دهم چرا بعضی چیزها را عمداً دست نزدم.

</div>


<div dir="rtl" style="text-align: right;">

### ۱. تنظیمات اولیه


</div>

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
np.random.seed(42)

def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for folder in [p, *p.parents]:
        if (folder / '.git').exists():
            return folder
    raise FileNotFoundError(f'.git not found above {p}')

REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / 'term-3' / 'datasets' / 'bank_midterm.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'dataset not found: {DATA_PATH}')

OUT_DIR = Path('data')
OUT_DIR.mkdir(exist_ok=True)
print(f'repo root: {REPO_ROOT}')
print(f'data file: {DATA_PATH}')

repo root: /home/user/dotin
data file: /home/user/dotin/term-3/datasets/bank_midterm.csv


<div dir="rtl" style="text-align: right;">



## ۲. خواندن داده و نگاه کلی
</div>

In [2]:
raw = pd.read_csv(DATA_PATH)
df = raw.copy()

print(f'shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

shape: 400 rows, 14 columns


,id,age,income,tenure_months,n_products,savings,investments,total_assets,region,debt_ratio,late_payments,legal_case,monthly_spend,default
0,1,41,34.5,22,5,216.7,22.6,239.3,Tehran,0.179,1,1,7.77,1
1,2,59,26.2,30,6,158.4,36.4,194.8,Tehran,0.323,2,0,6.44,0
2,3,56,17.5,60,2,107.9,22.2,130.1,Metro,0.327,1,0,3.97,0
3,4,58,17.4,102,3,104.2,34.4,138.6,Other,0.422,0,0,1.48,0
4,5,64,27.2,35,1,180.4,20.4,200.8,Tehran,0.661,0,0,4.17,0


In [3]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'unique': df.nunique(),
})
summary

,dtype,missing,unique
id,int64,0,400
age,int64,0,44
income,float64,0,271
tenure_months,int64,0,109
n_products,int64,0,8
savings,float64,0,378
investments,float64,0,341
total_assets,float64,0,376
region,str,0,3
debt_ratio,float64,0,288


<div dir="rtl" style="text-align: right;">

## ۳. چک‌های کیفیت داده
</div>

In [4]:
checks = {
    'missing values (total)': int(df.isna().sum().sum()),
    'duplicate rows (without id)': int(df.drop(columns='id').duplicated().sum()),
    'id is unique': bool(df['id'].is_unique),
    'income <= 0': int((df['income'] <= 0).sum()),
    'monthly_spend <= 0': int((df['monthly_spend'] <= 0).sum()),
    'tenure_months <= 0': int((df['tenure_months'] <= 0).sum()),
    'debt_ratio outside [0, 1]': int((~df['debt_ratio'].between(0, 1)).sum()),
    'age outside [18, 100]': int((~df['age'].between(18, 100)).sum()),
    'late_payments outside [0, 12]': int((~df['late_payments'].between(0, 12)).sum()),
    'legal_case not in {0,1}': int((~df['legal_case'].isin([0, 1])).sum()),
    'default not in {0,1}': int((~df['default'].isin([0, 1])).sum()),
}
pd.Series(checks, name='result').to_frame()

,result
missing values (total),0
duplicate rows (without id),0
id is unique,True
income <= 0,0
monthly_spend <= 0,0
tenure_months <= 0,0
"debt_ratio outside [0, 1]",0
"age outside [18, 100]",0
"late_payments outside [0, 12]",0
"legal_case not in {0,1}",0


In [5]:
print(df['region'].value_counts(dropna=False))
print()
print('labels with extra spaces:', int((df['region'] != df['region'].str.strip()).sum()))

region
Tehran    162
Other     121
Metro     117
Name: count, dtype: int64

labels with extra spaces: 0


<div dir="rtl" style="text-align: right;">

**نتیجه:** مقدار گمشده، سطر تکراری، مقدار غیرمنطقی و برچسب اشتباه در `region` وجود ندارد. داده از نظر ظاهری تمیز است. حالا دو چیز را بررسی می‌کنم که ممکن است مشکل ساز باشند ولی خطای ورود داده نیستند.

</div>

<div dir="rtl" style="text-align: right;">

## ۴. رابطه‌ی بین ستون‌های مالی
</div>

In [6]:
gap = (df['total_assets'] - df['savings'] - df['investments']).round(2)
print('max |total_assets - savings - investments|:', gap.abs().max())
print('rows where the identity breaks:', int((gap.abs() > 0.05).sum()))

print()
print('corr(income, savings):', round(df['income'].corr(df['savings']), 4))
print('savings / income  ->  mean:', round((df['savings'] / df['income']).mean(), 2),
      ' std:', round((df['savings'] / df['income']).std(), 2))

max |total_assets - savings - investments|: 0.0
rows where the identity breaks: 0

corr(income, savings): 0.9952
savings / income  ->  mean: 5.98  std: 0.41


<div dir="rtl" style="text-align: right;">

**نتیجه:**
- `total_assets` در هر ۴۰۰ سطر دقیقاً برابر `savings + investments` است. یعنی این ستون اطلاعات تازه‌ای ندارد.
- `savings` و `income` تقریباً یک رابطه‌ی خطی کامل دارند. این یعنی همخطی شدید.

این دو مورد خطای داده نیستند، ولی روی مدل رگرسیون اثر مستقیم دارند. در سوال ۱ باید با رتبه‌ی ماتریس طرح و VIF نشانشان بدهم، پس الان حذفشان نمی‌کنم.

</div>

<div dir="rtl" style="text-align: right;">

## ۵. مقدارهای دور از بقیه (پرت)
</div>

In [7]:
num_cols = ['income', 'savings', 'investments', 'tenure_months', 'monthly_spend']
rows = []
for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    far = df[(df[c] > q3 + 3 * iqr) | (df[c] < q1 - 3 * iqr)]
    rows.append({'column': c, 'n_far_values': len(far), 'max': df[c].max(), 'median': df[c].median()})
pd.DataFrame(rows)

,column,n_far_values,max,median
0,income,7,190.00,31.500
1,savings,7,1122.30,189.750
2,investments,8,332.50,43.850
3,tenure_months,0,128.00,35.000
4,monthly_spend,17,57.88,4.845


In [8]:
cols = ['id', 'income', 'savings', 'investments', 'monthly_spend']
df.nlargest(5, 'income')[cols]

,id,income,savings,investments,monthly_spend
310,311,190.0,1122.3,15.3,3.50
367,368,170.0,1002.1,47.8,2.50
31,32,160.0,958.9,7.5,39.45
151,152,150.0,917.5,85.7,3.00
83,84,109.0,658.7,110.4,57.88


In [9]:
df.nlargest(5, 'monthly_spend')[cols + ['region']]

,id,income,savings,investments,monthly_spend,region
83,84,109.0,658.7,110.4,57.88,Tehran
99,100,97.4,581.4,52.2,55.82,Tehran
355,356,25.1,142.8,63.5,45.00,Tehran
365,366,82.9,485.2,131.9,42.09,Tehran
397,398,58.9,360.1,57.4,40.72,Tehran


<div dir="rtl" style="text-align: right;">

**نتیجه:** چند مشتری درآمد و پس‌انداز خیلی بالا دارند و چند نفر هم خرج ماهانه‌ی خیلی بالا. این مقدارها ممکن و منطقی‌اند، یعنی نمی‌شود گفت اشتباه ثبت شده‌اند. فقط از بقیه‌ی داده دور هستند.

</div>

<div dir="rtl" style="text-align: right;">

## ۶. تغییرهای واقعی روی داده
</div>

In [10]:
df['region'] = pd.Categorical(df['region'], categories=['Tehran', 'Metro', 'Other'])

for c in ['n_products', 'late_payments', 'legal_case', 'default']:
    df[c] = df[c].astype(int)

print(df['region'].cat.categories.tolist())
print('rows before:', len(raw), ' rows after:', len(df))

['Tehran', 'Metro', 'Other']
rows before: 400  rows after: 400


In [11]:
df.to_csv(os.path.join(OUT_DIR, 'bank_clean.csv'), index=False)
print('saved: data/bank_clean.csv')

saved: data/bank_clean.csv


<div dir="rtl" style="text-align: right;">

## ۷. چرا بیشتر از این پولیش نکردم

**کاری که کردم:**
- `region` را به متغیر دسته‌ای تبدیل کردم و Tehran را سطح مرجع گذاشتم. بدون این کار، مدل ترتیب الفبایی را مرجع می‌گیرد و تفسیر ضریب‌ها سخت‌تر می‌شود.
- نوع ستون‌های شمارشی و ۰/۱ را عدد صحیح نگه داشتم.
- هیچ سطری حذف نشد (۴۰۰ سطر قبل و بعد).

**کارهایی که عمدا نکردم:**

۱. **حذف `total_assets` یا `savings`.** سوال ۱ از من می‌خواهد مدل را روی داده‌ی خام برازش بدهم و علت مشکل را با رتبه‌ی ماتریس و مقادیر ویژه نشان بدهم. اگر قبل از آن ستون را بردارم، چیزی برای نشان دادن نمی‌ماند.

۲. **حذف مقدارهای دور از بقیه.** سوال ۱ می‌خواهد مشاهدات تأثیرگذار را با اهرم، Cook's D و DFFITS تشخیص بدهم و بعد مدل را بدون آن‌ها برازش بدهم. اگر الان حذفشان کنم، این تشخیص بی‌معنی می‌شود. از طرف دیگر، دلیلی ندارد که این مقدارها اشتباه باشند، پس حذف بدون تشخیص درست نیست.

۳. **پر کردن یا تغییر مقدارها.** مقدار گمشده نداریم و هیچ مقداری خارج از محدوده‌ی منطقی نبود. هر تغییری فقط داده را از واقعیت دور می‌کرد.

۴. **تبدیل لگاریتمی یا استانداردسازی.** در سوال ۲ تبدیل مناسب را باید با Box–Cox و فاصله‌ی اطمینان انتخاب کنم. تبدیل زودهنگام جواب را از قبل تعیین می‌کند.

**خلاصه:** داده از نظر ورود اطلاعات تمیز است. مشکل‌هایش آماری‌اند (همخطی و نقاط تأثیرگذار) و خود سوال‌ها از من می‌خواهند آن‌ها را تشخیص بدهم و درست کنم. پس در این مرحله فقط داده را آماده می‌کنم و تشخیص و اصلاح را به بخش‌های بعدی می‌سپارم.

</div>

<div dir="rtl" style="text-align: right;">

## ۸. تحلیل اکتشافی (EDA)

هدف این بخش فقط شناخت داده است. هیچ تغییری روی داده نمی‌دهم و چیزی را حذف نمی‌کنم. چیزهایی که اینجا می‌بینم بعداً در سوال ۱ و ۲ به کار می‌آیند: چولگی (برای Box–Cox)، همخطی (برای VIF) و تغییر پراکندگی (برای ناهمسانی واریانس).

</div>

In [12]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

<div dir="rtl" style="text-align: right;">

### الف) توزیع پاسخ و درآمد

</div>

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['monthly_spend'], bins=40, kde=True, ax=axes[0], color='#4C72B0')
axes[0].set_title('monthly_spend')
sns.histplot(df['income'], bins=40, kde=True, ax=axes[1], color='#55A868')
axes[1].set_title('income')
plt.tight_layout()
plt.show()

print('skewness:')
print(df[['monthly_spend', 'income']].skew().round(2))
print()
print('skewness after log:')
print(np.log(df[['monthly_spend', 'income']]).skew().round(2))

skewness:
monthly_spend    4.42
income           3.58
dtype: float64

skewness after log:
monthly_spend    0.88
income           0.51
dtype: float64


<div dir="rtl" style="text-align: right;">

**توضیح:** هر دو توزیع به راست کشیده شده‌اند. چولگی `monthly_spend` برابر ۴٫۴۲ و `income` برابر ۳٫۵۸ است. بیشتر مشتری‌ها خرج کم دارند و چند نفر خیلی بالا هستند. بعد از لگاریتم چولگی کمتر می‌شود (۰٫۸۸ و ۰٫۵۱). این نشان می‌دهد در سوال ۲ تبدیل Box–Cox احتمالاً لازم می‌شود. فعلاً فقط یادداشتش می‌کنم و تبدیلی انجام نمی‌دهم.

</div>

<div dir="rtl" style="text-align: right;">

### ب) درآمد و پس‌انداز (همخطی)

</div>

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(data=df, x='income', y='savings', ax=axes[0], s=18, color='#C44E52')
axes[0].set_title('income vs savings')
sns.scatterplot(data=df, x='investments', y='total_assets', ax=axes[1], s=18, color='#8172B2')
axes[1].set_title('investments vs total_assets')
plt.tight_layout()
plt.show()

print('corr(income, savings):', round(df['income'].corr(df['savings']), 4))
print('corr(investments, total_assets):', round(df['investments'].corr(df['total_assets']), 4))

corr(income, savings): 0.9952
corr(investments, total_assets): 0.3768


<div dir="rtl" style="text-align: right;">

**توضیح:** نمودار سمت چپ تقریباً یک خط راست است (همبستگی ۰٫۹۹۵۲). یعنی `savings` تقریباً مضرب ثابتی از `income` است. نمودار سمت راست پراکنده است، چون `total_assets` از جمع `savings` و `investments` ساخته شده و به تنهایی با `investments` رابطه‌ی ضعیف‌تری دارد (۰٫۳۷۶۸). این رابطه‌ی ساختگی بعداً در سوال ۱-الف مشکل رتبه‌ی ماتریس طرح را می‌سازد.

</div>

<div dir="rtl" style="text-align: right;">

### ج) خرج ماهانه در برابر درآمد، به تفکیک منطقه

</div>

In [15]:
plt.figure(figsize=(7.5, 5))
sns.scatterplot(data=df, x='income', y='monthly_spend', hue='region', s=28, alpha=0.8)
plt.title('monthly_spend vs income')
plt.show()

df.groupby('region', observed=True)['monthly_spend'].agg(['count', 'median', 'mean', 'std']).round(2)

,count,median,mean,std
region,,,,
Tehran,162,7.17,10.10,8.93
Metro,117,4.12,4.54,2.08
Other,121,3.50,3.78,1.61


<div dir="rtl" style="text-align: right;">

**توضیح:** رابطه‌ی خرج و درآمد مثبت است ولی پراکندگی با بالا رفتن درآمد زیاد می‌شود (شکل شبیه قیف). بیشتر نقاط پرخرج مربوط به تهران هستند. میانه‌ی خرج در تهران ۷٫۱۷ است، در Metro برابر ۴٫۱۲ و در Other برابر ۳٫۵۰. انحراف معیار تهران (۸٫۹۳) چند برابر دو منطقه‌ی دیگر است. این الگو نشانه‌ی احتمالی ناهمسانی واریانس است و در سوال ۲-الف باید با آزمون Breusch–Pagan بررسی شود.

</div>

<div dir="rtl" style="text-align: right;">

### د) خرج ماهانه در هر منطقه

</div>

In [16]:
plt.figure(figsize=(7, 4.5))
sns.boxplot(data=df, x='region', y='monthly_spend', order=['Tehran', 'Metro', 'Other'])
plt.title('monthly_spend by region')
plt.show()

<div dir="rtl" style="text-align: right;">

**توضیح:** جعبه‌ی تهران هم بالاتر است و هم بازتر. ۵ مقدار بالای خرج (تا ۵۷٫۸۸) همگی مربوط به تهران‌اند. این یعنی `region` باید در مدل باشد، همان‌طور که سوال خواسته است.

</div>

<div dir="rtl" style="text-align: right;">

### ه) ماتریس همبستگی

</div>

In [17]:
num = df.select_dtypes('number').drop(columns='id')
corr = num.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': 0.8})
plt.title('correlation matrix')
plt.tight_layout()
plt.show()

print(corr['monthly_spend'].drop('monthly_spend').sort_values(ascending=False).round(2))

income           0.45
savings          0.45
total_assets     0.43
n_products       0.13
investments      0.03
late_payments   -0.02
tenure_months   -0.04
default         -0.07
legal_case      -0.07
debt_ratio      -0.09
age             -0.10
Name: monthly_spend, dtype: float64


<div dir="rtl" style="text-align: right;">

**توضیح:** قوی‌ترین همبستگی با `monthly_spend` مربوط به `income` و `savings` (هر دو ۰٫۴۵) و `total_assets` (۰٫۴۳) است. این سه خودشان به هم وابسته‌اند، پس اطلاعات مستقل زیادی نمی‌دهند. بقیه‌ی متغیرها (مثل `age` و `tenure_months`) همبستگی خطی ضعیفی دارند. همبستگی ضعیف یعنی رابطه‌ی خطی ضعیف است و نه این‌که اثری ندارند.

</div>

<div dir="rtl" style="text-align: right;">

### خلاصه‌ی بخش EDA

- `monthly_spend` و `income` چولگی راست زیادی دارند (احتمال نیاز به تبدیل در سوال ۲).
- `income` و `savings` تقریباً یکی هستند و `total_assets` مجموع `savings` و `investments` است (همخطی در سوال ۱).
- پراکندگی خرج با درآمد زیاد می‌شود و تهران بیشترین پراکندگی را دارد (ناهمسانی واریانس در سوال ۲).
- در این مرحله هیچ داده‌ای حذف یا تغییر نکرد.

</div>

<div dir="rtl" style="text-align: right;">

## ۹. سوال ۱-الف: مدل خام و همخطی کامل

مدل سوال را روی داده‌ی خام برازش می‌دهم، بدون هیچ حذف یا تغییری:

`monthly_spend ~ age + income + tenure_months + n_products + savings + investments + total_assets + region`

ماتریس طرح X شامل عرض از مبدأ، ۷ متغیر عددی و ۲ ستون dummy برای `region` است. پس ۱۰ ستون دارد. اگر همه‌ی ستون‌ها مستقل باشند، رتبه ۱۰ می‌شود.

</div>

In [18]:
import statsmodels.formula.api as smf

formula_raw = ('monthly_spend ~ age + income + tenure_months + n_products '
               '+ savings + investments + total_assets + C(region)')
m_raw = smf.ols(formula_raw, data=df).fit()

X = m_raw.model.exog
names = m_raw.model.exog_names
n, p = X.shape

print(f'n = {n}, p (columns of X) = {p}')
print(f'rank(X) = {np.linalg.matrix_rank(X)}')

n = 400, p (columns of X) = 10
rank(X) = 9


/tmp/ipykernel_1648/3015295052.py:5: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  m_raw = smf.ols(formula_raw, data=df).fit()


<div dir="rtl" style="text-align: right;">

**توضیح:** ماتریس طرح ۱۰ ستون دارد ولی رتبه‌اش فقط ۹ است. یعنی یک ستون ترکیب خطی بقیه است و `X'X` معکوس‌پذیر نیست. با این حال نرم‌افزار خطا نمی‌دهد و عدد می‌دهد. در ادامه می‌بینیم این عددها قابل اعتماد نیستند.

</div>

In [19]:
eigvals, eigvecs = np.linalg.eigh(X.T @ X)

print('eigenvalues of X\'X (smallest 4):')
print(eigvals[:4])
print()
print('largest eigenvalue:', f'{eigvals[-1]:.3e}')
print('condition number of X:', f'{np.linalg.cond(X):.3e}')
print()
print('eigenvector of the smallest eigenvalue:')
print(pd.Series(eigvecs[:, 0], index=names).round(3))

eigenvalues of X'X (smallest 4):
[-1.81529680e-09  1.27895655e+01  5.19900241e+01  1.16607526e+02]

largest eigenvalue: 6.183e+07
condition number of X: 5.423e+15

eigenvector of the smallest eigenvalue:
Intercept            -0.000
C(region)[T.Metro]    0.000
C(region)[T.Other]    0.000
age                   0.000
income                0.000
tenure_months        -0.000
n_products            0.000
savings              -0.577
investments          -0.577
total_assets          0.577
dtype: float64


<div dir="rtl" style="text-align: right;">

**توضیح:** کوچک‌ترین مقدار ویژه‌ی `X'X` عملاً صفر است (حدود ۱e-9 و حتی منفی، که فقط خطای گرد کردن کامپیوتر است). عدد شرط ماتریس حدود ۵٫۴e15 است. بردار ویژه‌ی متناظر فقط روی سه ستون مقدار دارد: `savings` و `investments` با وزن ‎−۰٫۵۷۷ و `total_assets` با وزن ‎+۰٫۵۷۷، و بقیه صفرند. یعنی رابطه‌ی خطی دقیق این است:

`total_assets − savings − investments = 0`

این همان چیزی است که در بخش ۴ دیدیم: `total_assets` همیشه برابر جمع دو ستون دیگر است.

</div>

In [20]:
pd.DataFrame({'coef': m_raw.params, 'se': m_raw.bse}).round(4)

,coef,se
Intercept,4.6857,1.3661
C(region)[T.Metro],-5.5405,0.6027
C(region)[T.Other],-6.6684,0.5971
age,-0.0734,0.0300
income,0.1777,0.1283
tenure_months,0.0256,0.0127
n_products,0.5916,0.1591
savings,-0.0049,0.0144
investments,0.0050,0.0079
total_assets,0.0001,0.0074


In [21]:
v = eigvecs[:, 0] / eigvecs[:, 0][names.index('total_assets')]
beta_alt = m_raw.params.values + 5 * v
fitted_alt = X @ beta_alt

print('max |fitted_original - fitted_shifted|:', np.abs(m_raw.fittedvalues.values - fitted_alt).max())
print('coef of savings      -> original:', round(m_raw.params['savings'], 4),
      ' shifted:', round(beta_alt[names.index('savings')], 4))
print('coef of total_assets -> original:', round(m_raw.params['total_assets'], 4),
      ' shifted:', round(beta_alt[names.index('total_assets')], 4))

max |fitted_original - fitted_shifted|: 1.3629719575192212e-10
coef of savings      -> original: -0.0049  shifted: -5.0049
coef of total_assets -> original: 0.0001  shifted: 5.0001


<div dir="rtl" style="text-align: right;">

**توضیح:** وقتی یک جهت صفر در `X'X` داریم، می‌توان به ضرایب هر مقدار دلخواه از آن جهت را اضافه کرد و مقدارهای برازش‌شده کاملاً یکسان می‌مانند. اینجا ضرایب را جابه‌جا کردم و اختلاف مقدارهای برازش‌شده صفر (در حد خطای گرد کردن) شد، ولی ضریب `savings` و `total_assets` عوض شد. نتیجه: بی‌نهایت جواب برای ضرایب وجود دارد و فقط پیش‌بینی ثابت است.

عددهایی که نرم‌افزار چاپ می‌کند یکی از همین بی‌نهایت جواب است. `statsmodels` از شبه‌معکوس استفاده می‌کند و جواب با کوچک‌ترین نُرم را برمی‌گرداند. پس این ضرایب و خطاهای استاندارد را نمی‌شود تفسیر کرد.

</div>

<div dir="rtl" style="text-align: right;">

### اصلاح مدل

برای رفع همخطی کامل، `total_assets` را حذف می‌کنم. چون مقدار آن دقیقاً از `savings` و `investments` ساخته می‌شود، حذفش هیچ اطلاعاتی را از دست نمی‌دهد. هر کدام از سه ستون را می‌شد حذف کرد، ولی `total_assets` انتخاب طبیعی است چون ستون مشتق‌شده است.

در سوال ۱-ب می‌بینیم که `income` و `savings` هم تقریباً یکی هستند. برای همین مدل نهایی بعد از آن مرحله `savings` را هم از دست می‌دهد و همان مدل «بدون `total_assets` و `savings`» می‌شود که سوال ۲ از آن استفاده می‌کند.

</div>

In [22]:
formula_fix = ('monthly_spend ~ age + income + tenure_months + n_products '
               '+ savings + investments + C(region)')
m_fix = smf.ols(formula_fix, data=df).fit()
X_fix = m_fix.model.exog

def summarize(model, X_mat):
    rank = np.linalg.matrix_rank(X_mat)
    return {
        'columns (p)': X_mat.shape[1],
        'rank(X)': rank,
        'nominal df = n - p': n - X_mat.shape[1],
        'true df = n - rank': n - rank,
        'smallest eigenvalue': f'{np.linalg.eigvalsh(X_mat.T @ X_mat)[0]:.2e}',
        'condition number': f'{np.linalg.cond(X_mat):.2e}',
        'RSS': round(model.ssr, 2),
        'R2': round(model.rsquared, 4),
    }

pd.DataFrame({
    'raw model': summarize(m_raw, X),
    'without total_assets': summarize(m_fix, X_fix),
})

,raw model,without total_assets
columns (p),10,9
rank(X),9,9
nominal df = n - p,390,391
true df = n - rank,391,391
smallest eigenvalue,-1.82e-09,1.28e+01
condition number,5.42e+15,1.44e+03
RSS,9555.01,9555.01
R2,0.4408,0.4408


In [23]:
pd.DataFrame({'coef': m_fix.params, 'se': m_fix.bse}).round(4)

,coef,se
Intercept,4.6857,1.3661
C(region)[T.Metro],-5.5405,0.6027
C(region)[T.Other],-6.6684,0.5971
age,-0.0734,0.0300
income,0.1777,0.1283
tenure_months,0.0256,0.0127
n_products,0.5916,0.1591
savings,-0.0047,0.0215
investments,0.0051,0.0052


<div dir="rtl" style="text-align: right;">

### درجه‌ی آزادی باقی‌مانده قبل و بعد از اصلاح

- **قبل از اصلاح:** اگر ستون‌ها را بشماریم، `n − p = 400 − 10 = 390`. ولی رتبه‌ی واقعی ۹ است، پس درجه‌ی آزادی واقعی `n − rank = 391` است. نرم‌افزار هم ۳۹۱ گزارش می‌کند، چون بر اساس رتبه حساب می‌کند.
- **بعد از اصلاح:** `p = rank = 9` و `df = 400 − 9 = 391`.

درجه‌ی آزادی در هر دو حالت ۳۹۱ است. فرق در تفسیر است: قبل از اصلاح، ۱۰ پارامتر داریم ولی فقط ۹ پارامتر مستقل قابل برآورد است. بعد از اصلاح، هر ۹ پارامتر یکتا برآورد می‌شوند.

### چه چیزی عوض شد و چه چیزی نه

- RSS و R² یکسان‌اند. پس قدرت پیش‌بینی مدل با حذف `total_assets` کمتر نشد. این همان نتیجه‌ی بخش قبل است: فقط ضرایب نامعین بودند، نه برازش.
- ضریب‌های مدل خام (مثل `savings` و `total_assets`) بی‌معنی بودند. در مدل اصلاح‌شده یکتا هستند.
- عدد شرط به‌شدت کاهش پیدا کرد. هنوز `income` و `savings` همخطی شدید دارند و این در بخش بعد (VIF) بررسی می‌شود.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۰. سوال ۱-ب: VIF و عدد شرط

در بخش ۹ مدل را از همخطی کامل نجات دادم (حذف `total_assets`). حالا بررسی می‌کنم آیا همخطی شدیدِ باقی‌مانده هم هست یا نه. مدل مبنا همان مدل بدون `total_assets` است.

</div>

In [24]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

formula_a = ('monthly_spend ~ age + income + tenure_months + n_products '
             '+ savings + investments + C(region)')
m_a = smf.ols(formula_a, data=df).fit()
X_a = m_a.model.exog
names_a = m_a.model.exog_names

vif_a = pd.Series(
    [variance_inflation_factor(X_a, i) for i in range(1, X_a.shape[1])],
    index=names_a[1:], name='VIF'
).round(2)
vif_a.sort_values(ascending=False).to_frame()

,VIF
savings,104.42
income,104.27
age,2.27
tenure_months,2.24
C(region)[T.Metro],1.23
C(region)[T.Other],1.23
n_products,1.02
investments,1.01


In [25]:
aux = smf.ols('income ~ age + tenure_months + n_products + savings + investments + C(region)', data=df).fit()
print('R2 of income regressed on the other predictors:', round(aux.rsquared, 4))
print('1 / (1 - R2) =', round(1 / (1 - aux.rsquared), 2), ' (same as VIF of income)')

R2 of income regressed on the other predictors: 0.9904
1 / (1 - R2) = 104.27  (same as VIF of income)


<div dir="rtl" style="text-align: right;">

**توضیح:** فقط دو متغیر مشکل دارند: `income` با VIF برابر ۱۰۴٫۲۷ و `savings` با ۱۰۴٫۴۲. قاعده‌ی معمول این است که VIF بالای ۱۰ نگران‌کننده است. بقیه‌ی متغیرها همه زیر ۲٫۳ هستند و مشکلی ندارند.

دلیل: VIF هر متغیر برابر `1 / (1 − R²)` است که R² از رگرسیون همان متغیر روی بقیه‌ی پیش‌بینی‌ها می‌آید. برای `income` این R² برابر ۰٫۹۹۰۴ است، یعنی ۹۹٪ تغییرات درآمد را `savings` توضیح می‌دهد. در بخش EDA هم دیدیم `savings` تقریباً ۶ برابر `income` است (نسبت میانگین ۵٫۹۸).

در مدل خام (با `total_assets`) VIF بی‌نهایت می‌شود، چون رابطه‌ی خطی دقیق است و R² برابر ۱ است.

</div>

In [26]:
def scaled_cond(X):
    Xs = X / np.linalg.norm(X, axis=0)
    s = np.linalg.svd(Xs, compute_uv=False)
    return s[0] / s[-1]

print('condition number of X (raw columns):', round(np.linalg.cond(X_a), 1))
print('condition number of X (columns scaled to unit length):', round(scaled_cond(X_a), 1))

Xs = X_a / np.linalg.norm(X_a, axis=0)
w, V = np.linalg.eigh(Xs.T @ Xs)
print()
print('smallest eigenvalues of scaled X\'X:', w[:3].round(5))
print('eigenvector of the smallest one:')
print(pd.Series(V[:, 0], index=names_a).round(3))

condition number of X (raw columns): 1438.9
condition number of X (columns scaled to unit length): 75.5

smallest eigenvalues of scaled X'X: [0.00113 0.01784 0.07382]
eigenvector of the smallest one:
Intercept             0.008
C(region)[T.Metro]    0.001
C(region)[T.Other]   -0.001
age                  -0.006
income                0.707
tenure_months        -0.000
n_products           -0.001
savings              -0.707
investments          -0.001
dtype: float64


<div dir="rtl" style="text-align: right;">

**توضیح:** عدد شرط ماتریس بعد از مقیاس‌دهی ستون‌ها (هر ستون با طول ۱) برابر ۷۵٫۵ است. قاعده‌ی معمول (Belsley) این است که بالای حدود ۳۰ نشانه‌ی همخطی جدی است، پس ۷۵٫۵ زیاد است. عدد شرط بدون مقیاس‌دهی (۱۴۳۸٫۹) را نباید به‌تنهایی قضاوت کرد، چون به واحد ستون‌ها وابسته است.

کوچک‌ترین مقدار ویژه حدود ۰٫۰۰۱۱ است و بردار ویژه‌ی آن تقریباً فقط روی `income` (‎+۰٫۷۰۷) و `savings` (‎−۰٫۷۰۷) مقدار دارد. یعنی مشکل دقیقاً به همان دو متغیر برمی‌گردد.

</div>

<div dir="rtl" style="text-align: right;">

### راه‌حل

`savings` را حذف می‌کنم و `income` را نگه می‌دارم. دلایل:
- `savings` تقریباً ۶ برابر `income` است و اطلاعات تازه‌ی زیادی ندارد.
- `income` متغیر ساده‌تر و قابل‌تفسیرتری برای خرج ماهانه است.
- سوال ۲ هم صریحاً می‌گوید از مدل بدون `total_assets` و `savings` استفاده کنیم.

**راه‌های دیگر و محدودیتشان:**
- حذف `income` به‌جای `savings`: کار می‌کند ولی تفسیر ضریب سخت‌تر می‌شود.
- Ridge یا Lasso: هر دو را نگه می‌دارد ولی ضرایب اریب می‌شوند و استنباط معمولی (SE و p-value) از بین می‌رود.
- ترکیب دو متغیر (مثلاً نسبت پس‌انداز به درآمد): اطلاعات را نگه می‌دارد ولی متغیر جدید می‌سازد.

</div>

In [27]:
formula_b = ('monthly_spend ~ age + income + tenure_months + n_products '
             '+ investments + C(region)')
m_b = smf.ols(formula_b, data=df).fit()
X_b = m_b.model.exog

vif_b = pd.Series(
    [variance_inflation_factor(X_b, i) for i in range(1, X_b.shape[1])],
    index=m_b.model.exog_names[1:], name='VIF'
).round(2)

print('max VIF before:', vif_a.max(), ' after:', vif_b.max())
print('scaled condition number before:', round(scaled_cond(X_a), 1), ' after:', round(scaled_cond(X_b), 1))
vif_b.sort_values(ascending=False).to_frame()

max VIF before: 104.42  after: 2.27
scaled condition number before: 75.5  after: 17.8


,VIF
age,2.27
tenure_months,2.24
C(region)[T.Metro],1.23
C(region)[T.Other],1.23
income,1.02
n_products,1.02
investments,1.01


In [28]:
before = pd.DataFrame({'coef': m_a.params, 'se': m_a.bse, 't': m_a.tvalues, 'p': m_a.pvalues})
after = pd.DataFrame({'coef': m_b.params, 'se': m_b.bse, 't': m_b.tvalues, 'p': m_b.pvalues})
cmp = before.join(after, lsuffix='_before', rsuffix='_after').round(4)
cmp

,coef_before,se_before,t_before,p_before,coef_after,se_after,t_after,p_after
Intercept,4.6857,1.3661,3.4299,0.0007,4.6674,1.3619,3.4270,0.0007
C(region)[T.Metro],-5.5405,0.6027,-9.1930,0.0000,-5.5449,0.6016,-9.2165,0.0000
C(region)[T.Other],-6.6684,0.5971,-11.1686,0.0000,-6.6663,0.5963,-11.1800,0.0000
age,-0.0734,0.0300,-2.4475,0.0148,-0.0731,0.0299,-2.4439,0.0150
income,0.1777,0.1283,1.3850,0.1668,0.1494,0.0127,11.7794,0.0000
tenure_months,0.0256,0.0127,2.0161,0.0445,0.0256,0.0127,2.0190,0.0442
n_products,0.5916,0.1591,3.7196,0.0002,0.5921,0.1588,3.7275,0.0002
savings,-0.0047,0.0215,-0.2211,0.8251,NaN,NaN,NaN,NaN
investments,0.0051,0.0052,0.9854,0.3250,0.0051,0.0052,0.9893,0.3231


In [29]:
print('income:')
print(f"  SE before = {m_a.bse['income']:.4f}   SE after = {m_b.bse['income']:.4f}   ratio = {m_a.bse['income'] / m_b.bse['income']:.1f}")
print(f"  t  before = {m_a.tvalues['income']:.2f}   t  after = {m_b.tvalues['income']:.2f}")
print(f"  95% CI before = {m_a.conf_int().loc['income'].round(3).tolist()}")
print(f"  95% CI after  = {m_b.conf_int().loc['income'].round(3).tolist()}")
print()
print('sqrt(VIF) of income before:', round(np.sqrt(vif_a['income']), 2))

income:
  SE before = 0.1283   SE after = 0.0127   ratio = 10.1
  t  before = 1.39   t  after = 11.78
  95% CI before = [-0.075, 0.43]
  95% CI after  = [0.125, 0.174]

sqrt(VIF) of income before: 10.21


<div dir="rtl" style="text-align: right;">

**توضیح:** بعد از حذف `savings`، خطای استاندارد ضریب `income` از ۰٫۱۲۸۳ به ۰٫۰۱۲۷ می‌رسد، یعنی حدود ۱۰ برابر کوچک‌تر. این دقیقاً با √VIF = √۱۰۴٫۲۷ ≈ ۱۰٫۲ جور است.

- **قبل:** t برابر ۱٫۳۹ و p برابر ۰٫۱۶۷ است. فاصله‌ی اطمینان ۹۵٪ از ‎−۰٫۰۷۵ تا ۰٫۴۳۰ می‌رود و صفر داخلش است. یعنی `income` ظاهراً بی‌اثر است.
- **بعد:** t برابر ۱۱٫۷۸ و p حدود ۱e‑۲۷ است. `income` به‌شدت معنادار است.

این نشان می‌دهد برآورد تفکیکی ضریب `income` زیر همخطی شدید ناپایدار است. با حذف `savings`، ضریب جدید بخشی از association مشترک درآمد و پس‌انداز را جذب می‌کند؛ این تغییر به‌تنهایی اثر علّی یا «اثر واقعی» درآمد را ثابت نمی‌کند.

</div>

In [30]:
slope = np.polyfit(df['income'], df['savings'], 1)[0]
implied = m_a.params['income'] + slope * m_a.params['savings']

print(f'slope of savings on income: {slope:.3f}')
print(f"income coef + slope * savings coef (before) = {implied:.4f}")
print(f"income coef after dropping savings          = {m_b.params['income']:.4f}")
print()
print('fitted values: max difference =', round(np.abs(m_a.fittedvalues - m_b.fittedvalues).max(), 4))
print(f'R2 before = {m_a.rsquared:.4f}, after = {m_b.rsquared:.4f}')
print(f'adj R2 before = {m_a.rsquared_adj:.4f}, after = {m_b.rsquared_adj:.4f}')
print(f'MSE before = {m_a.mse_resid:.3f}, after = {m_b.mse_resid:.3f}')
print()
print(m_a.f_test('income = 0, savings = 0'))

slope of savings on income: 5.954
income coef + slope * savings coef (before) = 0.1494
income coef after dropping savings          = 0.1494

fitted values: max difference = 0.1713
R2 before = 0.4408, after = 0.4408
adj R2 before = 0.4294, after = 0.4308
MSE before = 24.437, after = 24.378

<F test: F=69.23354090035608, p=1.8198624858624826e-26, df_denom=391, df_num=2>


<div dir="rtl" style="text-align: right;">

### همخطی چه چیزهایی را خراب می‌کند و چه چیزهایی را نه

**خراب می‌کند:**
- **خطای استاندارد، t و p-value** متغیرهای همخط. در مثال ما SE ضریب `income` حدود ۱۰ برابر شد و p از ۰٫۱۷ آمد.
- **پایداری ضرایب.** ضریب `savings` برابر ‎−۰٫۰۰۴۷ است (منفی و بی‌معنی)، در حالی که پس‌انداز بیشتر باید با خرج بیشتر همراه باشد. با کمی تغییر در داده این علامت‌ها به‌راحتی عوض می‌شوند.
- **تفسیر تک‌تک ضرایب.** «اثر `income` وقتی `savings` ثابت است» عملاً بی‌معنی است، چون این دو با هم تغییر می‌کنند.

**خراب نمی‌کند:**
- **مقدارهای برازش‌شده و پیش‌بینی.** اختلاف حداکثر برازش‌ها بین دو مدل ناچیز است (R² برابر ۰٫۴۴۰۸ و ۰٫۴۴۰۸). adj R² حتی کمی بهتر شد (۰٫۴۲۹۴ به ۰٫۴۳۰۸).
- **MSE (برآورد σ²):** ۲۴٫۴۴ و ۲۴٫۳۸، تقریباً یکسان.
- **ضرایب متغیرهای غیرهمخط:** `age`، `tenure_months`، `n_products`، `investments` و `region` هر دو بار تقریباً یکسان‌اند و SE‌شان هم تغییر چشمگیری نمی‌کند.
- **آزمون مشترک دو متغیر.** F برای فرضیه‌ی «ضریب `income` و `savings` هر دو صفر» برابر ۶۹٫۲ و p حدود ۱e‑۲۶ است. یعنی با هم معنادارند، ولی نمی‌شود سهم هر کدام را جدا کرد.

**چرا ضریب `income` عوض شد؟** ضریب بعد از حذف (۰٫۱۴۹۴) تقریباً برابر `income` + شیب × `savings` در مدل قبل است (۰٫۱۴۹۷). یعنی `income` اثر `savings` را هم جذب کرده است. پس در مدل جدید `income` نماینده‌ی هر دو (درآمد و ثروت) است و تفسیرش باید این را بگوید.

**نتیجه:** حذف `savings` در این داده برازش و پیش‌بینی را تقریباً تغییر نمی‌دهد و هم‌خطیِ ضرایب را کم می‌کند. بااین‌حال، ضریب درآمد در مدل کاهش‌یافته کمیت متفاوتی را توصیف می‌کند و نباید آن را اثر علّی یا اثر درآمد با ثابت بودن پس‌انداز تعبیر کرد.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۱. سوال ۱-ج: ماتریس هت و نقاط تأثیرگذار

**مدل مورد استفاده:** مدل اصلاح‌شده‌ی بخش ۱۰ (بدون `total_assets` و `savings`). دلیل: در مدل خام ماتریس `X'X` معکوس‌پذیر نیست و ماتریس هت درست ساخته نمی‌شود. مدل بدون `total_assets` هم همخطی شدید `income` و `savings` را دارد که اهرم‌ها را تحت تأثیر قرار می‌دهد. همچنین سوال ۲ همین مدل را می‌خواهد.

`monthly_spend ~ age + income + tenure_months + n_products + investments + region`

در این مدل `n = 400` و `p = 8` است (عرض از مبدأ، ۵ متغیر عددی و ۲ dummy).

</div>

In [31]:
formula_c = ('monthly_spend ~ age + income + tenure_months + n_products '
             '+ investments + C(region)')
m_c = smf.ols(formula_c, data=df).fit()

X = m_c.model.exog
y = df['monthly_spend'].to_numpy()
n, p = X.shape

# hat matrix built directly from matrices: H = X (X'X)^-1 X'
XtX_inv = np.linalg.inv(X.T @ X)
H = X @ XtX_inv @ X.T
h = np.diag(H)

print(f'n = {n}, p = {p}')
print('H is symmetric:', np.allclose(H, H.T))
print('H is idempotent (H @ H = H):', np.allclose(H @ H, H))
print(f'trace(H) = {np.trace(H):.6f}  (p = {p})')
print(f'leverage range: {h.min():.4f} to {h.max():.4f}')

n = 400, p = 8
H is symmetric: True
H is idempotent (H @ H = H): True
trace(H) = 8.000000  (p = 8)
leverage range: 0.0075 to 0.1686


<div dir="rtl" style="text-align: right;">

**توضیح:** ماتریس هت را مستقیم از `H = X(X'X)⁻¹X'` ساختم. سه ویژگی مورد انتظار برقرار است: متقارن است، خودتوان است (`H² = H`) و رد آن برابر ۸ است، یعنی دقیقاً `p`. این هم نتیجه‌ی خود فرمول است، چون `tr(H) = tr((X'X)⁻¹X'X) = tr(I_p) = p`. میانگین اهرم‌ها `p/n = 0.02` است.

</div>

In [32]:
threshold = 2 * p / n
high_lev = h > threshold

print(f'threshold 2p/n = {threshold:.4f}')
print(f'observations with h_ii > 2p/n: {high_lev.sum()}')
(pd.DataFrame({'id': df['id'], 'h_ii': h, 'income': df['income'], 'region': df['region']})
   .loc[high_lev].sort_values('h_ii', ascending=False).round(4))

threshold 2p/n = 0.0400
observations with h_ii > 2p/n: 16


,id,h_ii,income,region
310,311,0.1686,190.0,Other
367,368,0.1281,170.0,Other
31,32,0.1116,160.0,Tehran
151,152,0.0966,150.0,Other
50,51,0.0907,54.2,Other
160,161,0.0899,47.4,Other
204,205,0.0679,24.7,Tehran
262,263,0.0661,33.1,Metro
325,326,0.0591,36.7,Tehran
150,151,0.0588,43.0,Other


<div dir="rtl" style="text-align: right;">

**توضیح:** آستانه‌ی `2p/n = 0.04` است و ۱۶ مشاهده از آن بالاترند. بالاترین اهرم‌ها مال مشتری‌هایی با درآمد خیلی بالاست (مثل `id` 311 با درآمد ۱۹۰). این با EDA جور است: درآمد چولگی راست زیادی دارد و مشتری‌های پردرآمد از بقیه‌ی داده دورند.

</div>

In [33]:
e = y - H @ y                      # ordinary residuals
sse = e @ e
mse = sse / (n - p)

# externally studentized (deleted) residual
t_del = e * np.sqrt((n - p - 1) / (sse * (1 - h) - e**2))

# Cook's distance  D_i = e_i^2 / (p * MSE) * h_ii / (1 - h_ii)^2
cook = e**2 / (p * mse) * h / (1 - h)**2

# DFFITS  = t_i * sqrt(h_ii / (1 - h_ii))
dffits = t_del * np.sqrt(h / (1 - h))

res = pd.DataFrame({'id': df['id'], 'region': df['region'], 'income': df['income'],
                    'monthly_spend': df['monthly_spend'],
                    'h_ii': h, 't_deleted': t_del, 'cooks_D': cook, 'dffits': dffits})

# check against the library (only a check, not used to build anything)
infl = m_c.get_influence()
print('h matches library      :', np.allclose(h, infl.hat_matrix_diag))
print('Cook matches library   :', np.allclose(cook, infl.cooks_distance[0]))
print('DFFITS matches library :', np.allclose(dffits, infl.dffits[0]))
print('t_deleted matches      :', np.allclose(t_del, infl.resid_studentized_external))

h matches library      : True
Cook matches library   : True


DFFITS matches library : True
t_deleted matches      : True


<div dir="rtl" style="text-align: right;">

**توضیح:** همه‌ی کمیت‌ها را با فرمول خودشان حساب کردم و فقط برای اطمینان با کتابخانه مقایسه کردم. هر چهار یکسان درآمد. برای Cook از فرمول استاندارد با `p · MSE` استفاده کردم، نه `MSE²` که در صورت سوال آمده بود.

</div>

In [34]:
cut_t = 3
cut_cook = 4 / n
cut_dffits = 2 * np.sqrt(p / n)

res['leverage'] = res['h_ii'] > threshold
res['outlier'] = res['t_deleted'].abs() > cut_t
res['cook_big'] = res['cooks_D'] > cut_cook
res['dffits_big'] = res['dffits'].abs() > cut_dffits
res['influential'] = res['leverage'] & res['cook_big']

print(f'cutoffs: h > {threshold:.3f},  |t| > {cut_t},  Cook > 4/n = {cut_cook:.4f},  |DFFITS| > {cut_dffits:.3f}')
print()
print('leverage points  :', int(res['leverage'].sum()))
print('outliers         :', int(res['outlier'].sum()))
print('Cook > 4/n       :', int(res['cook_big'].sum()))
print('|DFFITS| large   :', int(res['dffits_big'].sum()))
print('influential      :', int(res['influential'].sum()), ' (leverage AND Cook large)')
print('max Cook D       :', round(res['cooks_D'].max(), 3), ' (no value above 1)')
print()
print('leverage only    :', int((res['leverage'] & ~res['outlier'] & ~res['influential']).sum()))
print('outlier only     :', int((res['outlier'] & ~res['leverage']).sum()))
print('leverage+outlier :', int((res['leverage'] & res['outlier']).sum()))

cutoffs: h > 0.040,  |t| > 3,  Cook > 4/n = 0.0100,  |DFFITS| > 0.283

leverage points  : 16
outliers         : 9
Cook > 4/n       : 12
|DFFITS| large   : 12
influential      : 5  (leverage AND Cook large)
max Cook D       : 0.656  (no value above 1)

leverage only    : 11
outlier only     : 5
leverage+outlier : 4


In [35]:
res.loc[res['leverage'] | res['outlier'] | res['cook_big']].sort_values('cooks_D', ascending=False).head(12).round(3)

,id,region,income,monthly_spend,h_ii,t_deleted,cooks_D,dffits,leverage,outlier,cook_big,dffits_big,influential
310,311,Other,190.0,3.50,0.169,-5.255,0.656,-2.367,True,True,True,True,True
367,368,Other,170.0,2.50,0.128,-4.879,0.413,-1.870,True,True,True,True,True
83,84,Tehran,109.0,57.88,0.053,7.724,0.366,1.836,True,True,True,True,True
99,100,Tehran,97.4,55.82,0.032,8.041,0.229,1.458,False,True,True,True,False
151,152,Other,150.0,3.00,0.097,-4.195,0.226,-1.372,True,True,True,True,True
365,366,Tehran,82.9,42.09,0.034,5.516,0.126,1.040,False,True,True,True,False
397,398,Tehran,58.9,40.72,0.035,5.032,0.107,0.954,False,True,True,True,False
31,32,Tehran,160.0,39.45,0.112,2.480,0.095,0.879,True,False,True,True,True
355,356,Tehran,25.1,45.00,0.010,7.801,0.069,0.797,False,True,True,True,False
185,186,Tehran,62.1,32.62,0.024,3.582,0.038,0.559,False,True,True,True,False


In [36]:
plt.figure(figsize=(8, 5.5))
plt.scatter(res['h_ii'], res['t_deleted'], s=20 + 600 * res['cooks_D'], alpha=0.45, color='#4C72B0')
inf = res[res['influential']]
plt.scatter(inf['h_ii'], inf['t_deleted'], s=20 + 600 * inf['cooks_D'], color='#C44E52', alpha=0.9, label='influential')
for _, r in inf.iterrows():
    plt.annotate(int(r['id']), (r['h_ii'], r['t_deleted']), textcoords='offset points', xytext=(6, 4))
plt.axvline(threshold, ls='--', color='gray')
plt.axhline(3, ls=':', color='gray')
plt.axhline(-3, ls=':', color='gray')
plt.xlabel('leverage h_ii')
plt.ylabel('deleted studentized residual')
plt.title('leverage vs residual (point size = Cook D)')
plt.legend()
plt.show()

<div dir="rtl" style="text-align: right;">

### دسته‌بندی

- **اهرمی (۱۶ مشاهده):** `h_ii > 0.04`. فاصله‌شان در فضای پیش‌بینی‌ها از بقیه زیاد است.
- **پرت (۹ مشاهده):** `|t حذفی| > 3`. باقی‌مانده‌شان بزرگ است. ۵ مشاهده فقط پرت‌اند (مثل `id` 100 و 366 و 398 و 356 و 186، همگی تهران با خرج خیلی بالا) و ۴ مشاهده هم اهرمی‌اند.
- **تأثیرگذار (۵ مشاهده):** هم اهرم بزرگ و هم `Cook > 4/n`. این‌ها `id` 311 و 368 و 84 و 152 و 32 هستند.

۱۱ مشاهده فقط اهرمی‌اند و اثر چندانی روی برازش ندارند. دلیلش این است که روی همان روند بقیه‌ی داده قرار دارند، پس باقی‌مانده‌شان کوچک است و Cook هم کوچک می‌ماند.

**نکات:**
- `id` 311 و 368 و 152: درآمد خیلی بالا (۱۹۰ و ۱۷۰ و ۱۵۰) با خرج خیلی کم، همگی منطقه‌ی Other. این‌ها شیب `income` را به پایین می‌کشند.
- `id` 84: خرج ۵۷٫۸۸، بالاترین خرج کل داده، و درآمد نسبتاً بالا (۱۰۹). باقی‌مانده‌اش مثبت و بزرگ است (t حدود ۷٫۷).
- `id` 32: درآمد ۱۶۰ و خرج ۳۹٫۴۵. اهرم بالا دارد ولی پرت نیست (t حدود ۲٫۵) و Cook آن فقط کمی بالای آستانه است.
- بیشترین مقدار Cook برابر ۰٫۶۵۶ است و هیچ مشاهده‌ای از ۱ بالاتر نیست. پس طبق قاعده‌ی سخت‌گیرانه‌ی `D > 1` هیچ نقطه‌ای مشکل‌ساز نیست. من از آستانه‌ی `4/n` استفاده کردم تا نقاط حساس‌تر دیده شوند، و گزارش می‌دهم که این آستانه سخت‌گیرانه‌تر است.

</div>

In [37]:
keep = ~res['influential'].to_numpy()
m_drop = smf.ols(formula_c, data=df.loc[keep]).fit()

change = pd.DataFrame({
    'coef_full': m_c.params,
    'coef_drop': m_drop.params,
    'se_full': m_c.bse,
    'se_drop': m_drop.bse,
})
change['change_%'] = ((change['coef_drop'] - change['coef_full']) / change['coef_full'].abs() * 100)
change['change_in_SE_units'] = (change['coef_drop'] - change['coef_full']) / change['se_full']
print(f'rows: {int(m_c.nobs)} -> {int(m_drop.nobs)}')
print(f'R2: {m_c.rsquared:.4f} -> {m_drop.rsquared:.4f}')
print(f'MSE: {m_c.mse_resid:.2f} -> {m_drop.mse_resid:.2f}')
change.round(4)

rows: 400 -> 395
R2: 0.4408 -> 0.5132
MSE: 24.38 -> 16.76


,coef_full,coef_drop,se_full,se_drop,change_%,change_in_SE_units
Intercept,4.6674,1.8570,1.3619,1.1896,-60.2124,-2.0635
C(region)[T.Metro],-5.5449,-5.3398,0.6016,0.5003,3.6985,0.3409
C(region)[T.Other],-6.6663,-5.6983,0.5963,0.4998,14.5200,1.6233
age,-0.0731,-0.0613,0.0299,0.0249,16.2115,0.3962
income,0.1494,0.2133,0.0127,0.0148,42.6935,5.0291
tenure_months,0.0256,0.0262,0.0127,0.0105,2.1102,0.0426
n_products,0.5921,0.5774,0.1588,0.1326,-2.4867,-0.0927
investments,0.0051,0.0023,0.0052,0.0043,-55.5435,-0.5495


<div dir="rtl" style="text-align: right;">

### مدل بدون نقاط تأثیرگذار

بعد از حذف ۵ مشاهده‌ی تأثیرگذار، تعداد سطرها ۳۹۵ است.

- **`income`:** ضریب از ۰٫۱۴۹۴ به ۰٫۲۱۳۳ می‌رود، حدود ۴۳٪ بیشتر. این جابه‌جایی حدود ۵ برابر SE مدل کامل است و حساسیت برازش به این پنج مشاهده را نشان می‌دهد؛ به‌تنهایی ثابت نمی‌کند که آن مشاهدات خطا هستند یا باید حذف شوند. چهار تا از این پنج مشاهده مشتری‌های پردرآمدی بودند که خرج کمی داشتند و شیب را پایین می‌کشیدند.
- **`Intercept`:** از ۴٫۶۷ به ۱٫۸۶ می‌رسد (حدود ۶۰٪ کمتر).
- **`investments`:** ضریب خیلی کوچک بود و قبل و بعد معنادار نیست (t حدود ۰٫۵).
- **`region` (Other):** از ‎−۶٫۶۷ به ‎−۵٫۷۰ تغییر می‌کند.
- **بقیه** (`age`، `tenure_months`، `n_products`، `region` Metro) تغییر کمی دارند.
- **R²** از ۰٫۴۴۱ به ۰٫۵۱۳ و **MSE** از ۲۴٫۳۸ به ۱۶٫۷۶ می‌رسد. حذف فقط ۱٫۲۵٪ داده، برازش را به‌طور محسوس بهتر کرد.

این ۵ مشاهده روی نتیجه‌ی مدل اثر زیادی دارند، مخصوصاً روی `income`. با این حال دلیل نمی‌شود که اشتباه ثبت شده باشند. در بخش ۱-د نشان می‌دهم چرا حذف خودکار نقاط اهرمی درست نیست. در گزارش هر دو مدل را کنار هم گزارش می‌کنم و به خواننده می‌گویم نتیجه به این ۵ نفر حساس است.

</div>

<div dir="rtl" style="text-align: right;">

## سوال ۱-د (مفهومی): اهرم، PRESS و Cook

در این بخش فرض می‌کنیم ماتریس طرح `X` رتبه‌ی کامل ستونی دارد و `X` را هنگام مشتق‌گیری ثابت نگه می‌داریم. در انتهای این پاسخ، اسکن یادداشت دست‌نویس نیز آمده است؛ اثبات بند (ii) را برای رعایت شرط تحویل دستی به برگه‌ی دست‌نویس اضافه کنید.

### i) مشتق مقدار برازش‌شده و بازه‌ی اهرم

`H = X(X'X)^{-1}X'` و `ŷ = Hy`، پس

\[
\hat y_i=\sum_j h_{ij}y_j
\quad\Longrightarrow\quad
\frac{\partial\hat y_i}{\partial y_i}=h_{ii}.
\]

از متقارن و خودتوان بودن `H` داریم:

\[
h_{ii}=(H^2)_{ii}=\sum_j h_{ij}^2
=h_{ii}^2+\sum_{j\ne i}h_{ij}^2,
\]

بنابراین `h_ii(1-h_ii) = Σ(j≠i) h_ij² ≥ 0` و در نتیجه `0 ≤ h_ii ≤ 1`.

### ii) باقیمانده‌ی حذفِ مشاهده (PRESS)

بگذارید `A = X'X`، سطر `i` ماتریس طرح را `x_i'` بنامیم، و `e_i = y_i - ŷ_i`. پس از حذف مشاهده‌ی `i`، ماتریس نرمال برابر `A_(−i) = A - x_i x_i'` است. با فرمول Sherman–Morrison، برای `h_ii = x_i'A⁻¹x_i < 1`:

\[
\hat\beta_{(-i)}
=\hat\beta-\frac{A^{-1}x_i e_i}{1-h_{ii}}.
\]

پس `ŷ_(−i) = x_i'β̂_(−i) = ŷ_i - h_ii e_i/(1-h_ii)` و در نتیجه

\[
y_i-\hat y_{i(-i)}
=e_i+\frac{h_{ii}e_i}{1-h_{ii}}
=\boxed{\frac{e_i}{1-h_{ii}}}.
\]

### iii) چرا Cook's D به اهرم و باقیمانده هر دو وابسته است؟

فرمول استاندارد Cook برای OLS این است:

\[
D_i=\frac{e_i^2}{p\,\mathrm{MSE}}\frac{h_{ii}}{(1-h_{ii})^2}
=\frac{r_i^2}{p}\frac{h_{ii}}{1-h_{ii}},
\qquad r_i=\frac{e_i}{\sqrt{\mathrm{MSE}(1-h_{ii})}}.
\]

در صورت سؤال ظاهراً `MSE²` نوشته شده است؛ مخرج استاندارد `p·MSE` است. این انتخاب Cook's D را بی‌بعد می‌کند.

دو نمونه از داده با مدل سوال ۱-ج:
- مشاهده‌ی `id=51`: اهرم بالا (`h≈0.0907 > 0.04`) اما باقیمانده کوچک (`e≈−1.74`) است؛ بنابراین `D≈0.0017` کوچک می‌ماند.
- مشاهده‌ی `id=100`: اهرم پایین‌تر از آستانه (`h≈0.0318 < 0.04`) اما باقیمانده‌ی بزرگی دارد (`e≈36.23`)؛ بنابراین `D≈0.229` متوسط/قابل‌توجه است.

پس اهرم به‌تنهایی Cook's D بزرگ را تضمین نمی‌کند؛ باقیمانده‌ی بزرگ نیز می‌تواند با اهرم نه‌چندان زیاد به D قابل‌توجه منجر شود.

### iv) آیا هر مشاهده‌ی اهرم‌بالا باید حذف شود؟

خیر. اهرم‌بالا یعنی ترکیب پیش‌بین‌ها در فضای `X` غیرمعمول است، نه این‌که پاسخ اشتباه ثبت شده یا مشاهده الزاماً اثرگذار است. نقطه‌ای با اهرم بالا اما باقیمانده‌ی کوچک ممکن است Cook's D کوچکی داشته باشد. ابتدا داده و دامنه‌ی مشاهده بررسی شود؛ اگر معتبر است، آن را در تحلیل اصلی نگه دارید و حساسیت نتایج (مثلاً با و بدون مشاهده) را گزارش کنید. حذف خودکار می‌تواند اطلاعات واقعیِ بخش مهمی از جمعیت را دور بریزد.

</div>

![اسکن پاسخ دست‌نویس سوال ۱-د، صفحه ۱](images/1d_page1.jpg.jpg)

![اسکن پاسخ دست‌نویس سوال ۱-د، صفحه ۲](images/1d_page2.jpg.jpg)

---

<div dir="rtl" style="text-align: right;">

## ۱۲. سوال ۲-الف: فرضیات مدل

از مدل اصلاح‌شده‌ی سوال ۱ روی **همه‌ی ۴۰۰ سطر** استفاده می‌کنم (بدون `total_assets` و `savings`، و بدون حذف هیچ مشاهده‌ای):

`monthly_spend ~ age + income + tenure_months + n_products + investments + region`

فرض‌هایی که بررسی می‌شوند: واریانس ثابت (Breusch–Pagan و نمودار باقی‌مانده) و نرمال بودن خطا (Shapiro–Wilk و Q-Q).

</div>

In [38]:
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

formula_q2 = ('monthly_spend ~ age + income + tenure_months + n_products '
              '+ investments + C(region)')
m_q2 = smf.ols(formula_q2, data=df).fit()
X_q2 = m_q2.model.exog
resid = m_q2.resid
fitted = m_q2.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

axes[0].scatter(fitted, resid, s=18, alpha=0.6, color='#4C72B0')
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('fitted values')
axes[0].set_ylabel('residuals')
axes[0].set_title('residuals vs fitted')

sm.qqplot(resid, line='s', ax=axes[1], markersize=3)
axes[1].set_title('normal Q-Q plot')

plt.tight_layout()
plt.show()

In [39]:
bp_lm, bp_p, bp_f, bp_fp = het_breuschpagan(resid, X_q2)
sw_w, sw_p = stats.shapiro(resid)

print(f'Breusch-Pagan: LM = {bp_lm:.2f}, p = {bp_p:.2e}')
print(f'Shapiro-Wilk : W = {sw_w:.4f}, p = {sw_p:.2e}')
print(f'residual skewness = {stats.skew(resid):.2f}, excess kurtosis = {stats.kurtosis(resid):.2f}')
print(f'fitted values <= 0: {(fitted <= 0).sum()} (min = {fitted.min():.2f})')

Breusch-Pagan: LM = 89.26, p = 1.75e-16
Shapiro-Wilk : W = 0.6567, p = 1.34e-27
residual skewness = 3.09, excess kurtosis = 24.86
fitted values <= 0: 8 (min = -1.04)


In [40]:
by_region = (pd.DataFrame({'resid': resid, 'region': df['region']})
             .groupby('region', observed=True)['resid'].agg(['std', 'min', 'max']).round(2))
by_fitted = (pd.Series(resid.values).groupby(pd.qcut(fitted.values, 4), observed=True).std().round(2))
by_fitted.index = [f'Q{i + 1}' for i in range(4)]

print('residual SD by region:')
print(by_region)
print()
print('residual SD by fitted-value quartile:')
print(by_fitted.to_frame('SD'))

residual SD by region:
         std    min    max
region                    
Tehran  6.72  -6.39  36.23
Metro   1.84  -4.73   6.24
Other   3.98 -22.90   3.97

residual SD by fitted-value quartile:
      SD
Q1  1.26
Q2  1.51
Q3  4.05
Q4  8.37


<div dir="rtl" style="text-align: right;">

**نتیجه:** هر دو فرض نقض شده‌اند.

**ناهمسانی واریانس:** آزمون Breusch–Pagan رد می‌کند (LM برابر ۸۹٫۲۶ و p حدود ۱٫۸e‑۱۶). در نمودار، پراکندگی باقی‌مانده‌ها با بالا رفتن مقدار برازش‌شده زیاد می‌شود (شکل قیف یا بادبزن). انحراف معیار باقی‌مانده‌ها در چارک‌های مقدار برازش‌شده از ۱٫۲۶ و ۱٫۵۱ به ۴٫۰۵ و ۸٫۳۷ می‌رسد، یعنی حدود ۶ برابر. بر اساس منطقه هم فرق دارد: تهران ۶٫۷۲، Other ۳٫۹۸ و Metro ۱٫۸۴. ساختار واریانس تقریباً متناسب با سطح پاسخ است (هر چه خرج بیشتر، پراکندگی بیشتر).

**نرمال نبودن:** آزمون Shapiro–Wilk رد می‌کند (W برابر ۰٫۶۵۷ و p حدود ۱e‑۲۷). در Q-Q دنباله‌ی راست خیلی از خط بالاتر می‌رود. چولگی باقی‌مانده‌ها ۳٫۰۹ و کشیدگی اضافه ۲۴٫۸۶ است. یعنی توزیع به راست کج است و دنباله‌ی سنگین دارد.

**یک مشکل دیگر:** ۸ مقدار برازش‌شده صفر یا منفی‌اند (کمترین ‎−۱٫۰۴). خرج ماهانه نمی‌تواند منفی باشد، پس مدل خطی روی مقیاس اصلی در پایین‌ترین مقادیر درست کار نمی‌کند.

**فرض‌هایی که نقض نشده‌اند یا اینجا بررسی نمی‌شوند:** خطی بودن در نمودار باقی‌مانده انحراف شدیدی نشان نمی‌دهد (مرکز نمودار تقریباً حول صفر است). همخطی را در سوال ۱ حل کردیم. استقلال خطاها در داده‌ی مقطعی بدون ترتیب زمانی قابل بررسی نیست.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۳. سوال ۲-ب: تبدیل Box–Cox

پاسخ باید مثبت باشد (کمترین `monthly_spend` برابر ۰٫۸۹ است) و Box–Cox قابل استفاده است. λ را با درست‌نمایی نیم‌رخ (profile likelihood) **با در نظر گرفتن متغیرهای مدل** برآورد می‌کنم، نه فقط روی خود `y`:

\( L(\lambda) = -\frac{n}{2}\log\frac{RSS(\lambda)}{n} + (\lambda-1)\sum \log y_i \)

فاصله‌ی اطمینان ۹۵٪ همه‌ی λ هایی است که `L(λ)` از حداکثر بیشتر از \( \chi^2_{1,0.95}/2 = 1.92 \) پایین‌تر نباشد.

</div>

In [41]:
y = df['monthly_spend'].to_numpy()
n = len(y)
sum_log_y = np.log(y).sum()

def transform(lam):
    return np.log(y) if abs(lam) < 1e-9 else (y**lam - 1) / lam

def profile_ll(lam):
    z = transform(lam)
    b = np.linalg.lstsq(X_q2, z, rcond=None)[0]
    rss = ((z - X_q2 @ b) ** 2).sum()
    return -n / 2 * np.log(rss / n) + (lam - 1) * sum_log_y

grid = np.linspace(-1.5, 1.5, 601)
ll = np.array([profile_ll(l) for l in grid])
lam_hat = grid[ll.argmax()]
cutoff = ll.max() - stats.chi2.ppf(0.95, 1) / 2
inside = grid[ll >= cutoff]
lam_lo, lam_hi = inside.min(), inside.max()

print(f'lambda_hat = {lam_hat:.3f}')
print(f'95% CI     = [{lam_lo:.3f}, {lam_hi:.3f}]')

plt.figure(figsize=(7.5, 4.8))
plt.plot(grid, ll, color='#4C72B0')
plt.axhline(cutoff, color='gray', ls='--', lw=1)
plt.axvline(lam_hat, color='red', lw=1, label=f'lambda_hat = {lam_hat:.2f}')
plt.axvline(lam_lo, color='gray', ls=':', lw=1)
plt.axvline(lam_hi, color='gray', ls=':', lw=1)
plt.xlabel('lambda')
plt.ylabel('profile log-likelihood')
plt.title('Box-Cox profile log-likelihood')
plt.legend()
plt.show()

lambda_hat = -0.250
95% CI     = [-0.345, -0.160]


In [42]:
for lam, name in [(1, 'lambda = 1 (no transform)'), (0.5, 'lambda = 0.5'), (0, 'lambda = 0 (log)'),
                  (-0.25, 'lambda = -0.25'), (-0.5, 'lambda = -0.5')]:
    lr = 2 * (ll.max() - profile_ll(lam))
    print(f'{name:28s} LR = {lr:7.2f}   p = {stats.chi2.sf(lr, 1):.2e}   inside CI: {lr < 3.84}')

lambda = 1 (no transform)    LR =  719.44   p = 1.78e-158   inside CI: False
lambda = 0.5                 LR =  262.40   p = 5.14e-59   inside CI: False
lambda = 0 (log)             LR =   27.60   p = 1.49e-07   inside CI: False
lambda = -0.25               LR =    0.00   p = 1.00e+00   inside CI: True
lambda = -0.5                LR =   23.90   p = 1.02e-06   inside CI: False


<div dir="rtl" style="text-align: right;">

**نتیجه‌ی Box–Cox:** `λ̂ = −۰٫۲۵` و فاصله‌ی اطمینان ۹۵٪ تقریباً `[−۰٫۳۴۵ ، −۰٫۱۶۰]` است. `λ=1` و `λ=0` (لگاریتم) خارج از فاصله‌اند؛ برای `λ=0` آماره‌ی نسبت درست‌نمایی ۲۷٫۶ و `p≈1.5e−7` است. بنابراین، تبدیل انتخاب‌شده برای بخش **۲-ب** همان Box–Cox با `λ=−۰٫۲۵` است، نه لگاریتم.

صورت سؤال در بخش‌های ۲-ج و ۲-د جداگانه صراحتاً مدل لگاریتمی می‌خواهد. برای رفع این ناسازگاری، این بخش مدل Box–Cox انتخاب‌شده را برازش و تشخیص‌گذاری می‌کند؛ مدل لگاریتمی جداگانه در ادامه، فقط برای محاسباتی که خود سؤال log model خواسته است، استفاده می‌شود.

نکته: برآورد λ با پروفایل درست‌نماییِ مدل و متغیرهای پیش‌بین انجام شده است؛ برازش `scipy.stats.boxcox` روی پاسخِ تنها، مسئله‌ی متفاوتی را حل می‌کند.

</div>

In [43]:
def diagnose(z, label):
    b = np.linalg.lstsq(X_q2, z, rcond=None)[0]
    res = z - X_q2 @ b
    bp = het_breuschpagan(res, X_q2)
    sw = stats.shapiro(res)
    return {'model': label, 'BP LM': round(bp[0], 2), 'BP p': f'{bp[1]:.1e}',
            'SW W': round(sw[0], 4), 'SW p': f'{sw[1]:.1e}',
            'skew': round(stats.skew(res), 2), 'excess kurtosis': round(stats.kurtosis(res), 2)}

compare = pd.DataFrame([
    diagnose(y, 'original y'),
    diagnose(np.log(y), 'log(y)'),
    diagnose(transform(-0.25), 'lambda = -0.25'),
]).set_index('model')
compare

,BP LM,BP p,SW W,SW p,skew,excess kurtosis
model,,,,,,
original y,89.26,1.8e-16,0.6567,1.3e-27,3.09,24.86
log(y),124.31,9.7e-24,0.9520,4.0e-10,-0.40,4.17
lambda = -0.25,94.58,1.4e-17,0.9605,6.7e-09,-0.73,3.04


In [44]:
# Q2(b): fit the Box-Cox transformation selected by the profile likelihood.
df['monthly_spend_bc'] = transform(lam_hat)
formula_bc = formula_q2.replace('monthly_spend', 'monthly_spend_bc')
m_bc = smf.ols(formula_bc, data=df).fit()
res_bc = m_bc.resid
fit_bc = m_bc.fittedvalues

# Separate log model: Q2(c)-(d) explicitly request log-scale predictions/corrections.
m_log = smf.ols(formula_q2.replace('monthly_spend', 'np.log(monthly_spend)'), data=df).fit()
res_log = m_log.resid
fit_log = m_log.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].scatter(fit_bc, res_bc, s=18, alpha=0.6, color='#4C72B0')
axes[0, 0].axhline(0, color='red', lw=1)
axes[0, 0].set_xlabel('fitted (Box-Cox scale)')
axes[0, 0].set_ylabel('residuals')
axes[0, 0].set_title('selected Box-Cox model: residuals vs fitted')
sm.qqplot(res_bc, line='s', ax=axes[0, 1], markersize=3)
axes[0, 1].set_title('selected Box-Cox model: normal Q-Q')

axes[1, 0].scatter(fit_log, res_log, s=18, alpha=0.6, color='#55A868')
axes[1, 0].axhline(0, color='red', lw=1)
axes[1, 0].set_xlabel('fitted (log scale)')
axes[1, 0].set_ylabel('residuals')
axes[1, 0].set_title('separate log model: residuals vs fitted')
sm.qqplot(res_log, line='s', ax=axes[1, 1], markersize=3)
axes[1, 1].set_title('separate log model: normal Q-Q')
plt.tight_layout()
plt.show()

print('Box-Cox residual SD by region:')
print(pd.Series(res_bc.values).groupby(df['region'].values).std().round(3))
print('Log-model residual SD by region:')
print(pd.Series(res_log.values).groupby(df['region'].values).std().round(3))

Box-Cox residual SD by region:
Tehran    0.234
Metro     0.197
Other     0.325
dtype: float64
Log-model residual SD by region:
Tehran    0.417
Metro     0.280
Other     0.468
dtype: float64


<div dir="rtl" style="text-align: right;">

**تشخیص‌ها پس از تبدیل انتخاب‌شده (`λ=−۰٫۲۵`):**

| مدل | Breusch–Pagan (LM) | Shapiro–Wilk (W) | چولگی | کشیدگی اضافه |
|---|---:|---:|---:|---:|
| `y` اصلی | ۸۹٫۲۶ | ۰٫۶۵۷ | ۳٫۰۹ | ۲۴٫۸۶ |
| Box–Cox، `λ=−۰٫۲۵` | ۹۴٫۵۸ | ۰٫۹۶۰۵ | ‎−۰٫۷۳ | ۳٫۰۴ |
| لگاریتم (مدل جداگانه برای ۲-ج/د) | ۱۲۴٫۳۱ | ۰٫۹۵۲ | ‎−۰٫۴۰ | ۴٫۱۷ |

Box–Cox چولگی و شکل Q–Q را نسبت به مقیاس اصلی بهتر می‌کند، اما Shapiro–Wilk هنوز نرمالیت را رد می‌کند (`p≈6.7e−9`). این رد را صرفاً به «حساسیت آزمون» نسبت نمی‌دهم؛ نمودار Q–Q نیز باید بررسی شود. ناهمسانی واریانس هم باقی می‌ماند (BP برای Box–Cox `p≈1.4e−17`). نمودارهای سطر اول سلول قبل مربوط به تبدیل انتخاب‌شده‌اند؛ سطر دوم، مدل لگاریتمی جداگانه‌ای است که برای بخش‌های بعدیِ صریحاً log استفاده می‌شود.

</div>

In [45]:
bc_coef = m_bc.params['income']
bc_ci = m_bc.conf_int().loc['income']
log_coef = m_log.params['income']

print(f'Box-Cox income coefficient = {bc_coef:.6f}')
print(f'95% CI on Box-Cox scale = [{bc_ci.iloc[0]:.6f}, {bc_ci.iloc[1]:.6f}]')
print(f'Separate log-model coefficient = {log_coef:.6f}')
print(f'log-model multiplicative change per +1 income = {(np.exp(log_coef) - 1) * 100:.2f}%')

Box-Cox income coefficient = 0.007526
95% CI on Box-Cox scale = [0.006225, 0.008827]
Separate log-model coefficient = 0.012684
log-model multiplicative change per +1 income = 1.28%


<div dir="rtl" style="text-align: right;">

**تفسیر ضریب درآمد:** در مدل انتخاب‌شده‌ی Box–Cox، افزایش یک واحد `income` (یک میلیون تومان)، با ثابت بودن سایر پیش‌بین‌ها، میانگین پاسخِ تبدیل‌شده را به اندازه‌ی `۰٫۰۰۷۵۲۶` واحد روی مقیاس Box–Cox جابه‌جا می‌کند (فاصله‌ی اطمینان ۹۵٪: `۰٫۰۰۶۲۲۵` تا `۰٫۰۰۸۸۲۷`). این ضریب تغییر درصدیِ ثابت روی `monthly_spend` اصلی نیست؛ تبدیل معکوس غیرخطی است و اثر روی مقیاس اصلی به سطح پیش‌بینی‌شده نیز وابسته می‌شود.

برای مدل لگاریتمیِ جداگانه‌ای که بخش‌های ۲-ج/د درخواست کرده‌اند، ضریب درآمد `۰٫۰۱۲۶۸۴` است؛ در آن مدل، هر یک میلیون تومان درآمد بیشتر با حدود `۱٫۲۸٪` خرج بیشتر همراه است (`exp(β̂)−1`). این تفسیر به مدل لگاریتمی تعلق دارد و جایگزین تفسیر Box–Cox بالا نیست.

</div>

<div dir="rtl" style="text-align: right;">

## ۱۴. سوال ۲-ج: WLS و رگرسیون مقاوم

### تابع واریانس
از بخش الف دیدیم پراکندگی باقی‌مانده‌ها با سطح پاسخ بالا می‌رود. چند تابع واریانس را امتحان می‌کنم و بعد از وزن‌دهی دوباره Breusch–Pagan را روی باقی‌مانده‌های وزنی اجرا می‌کنم. هدف این است که واریانس تقریباً ثابت شود.

</div>

In [46]:
sq = resid ** 2

def weighted_check(w, label):
    fit_w = sm.WLS(y, X_q2, weights=w).fit()
    res_w = fit_w.resid * np.sqrt(w)
    lm, p, _, _ = het_breuschpagan(res_w, X_q2)
    return {'variance function': label, 'BP LM after WLS': round(lm, 1), 'BP p': f'{p:.1e}'}

# 1) variance by region
sd_region = sq.groupby(df['region'], observed=True).transform('mean').to_numpy()
w_region = 1 / sd_region

# 2) log(e^2) ~ region + log(income)
aux = smf.ols('np.log(sq) ~ C(region) + np.log(income)', data=df.assign(sq=sq)).fit()
w_aux = 1 / np.exp(aux.fittedvalues.to_numpy())

# 3) variance proportional to fitted^2 (constant coefficient of variation)
fit_floor = np.maximum(fitted.to_numpy(), 1.0)
w_fit2 = 1 / fit_floor ** 2

pd.DataFrame([
    weighted_check(np.ones(n), 'none (plain OLS)'),
    weighted_check(w_region, 'by region'),
    weighted_check(w_aux, 'region + log(income)'),
    weighted_check(w_fit2, 'Var proportional to max(fitted, 1)^2'),
]).set_index('variance function')

,BP LM after WLS,BP p
variance function,,
none (plain OLS),89.3,1.8e-16
by region,102.3,3.6e-19
region + log(income),19.8,6.1e-03
"Var proportional to max(fitted, 1)^2",6.0,5.4e-01


<div dir="rtl" style="text-align: right;">

**انتخاب اکتشافی تابع واریانس:** تابع `Var(εᵢ|X) ∝ μᵢ²` را بررسی می‌کنم؛ یعنی اگر میانگین مثبت باشد، انحراف معیار با میانگین متناسب است و ضریب تغییرات ثابت می‌ماند. چون ۲۴ مقدار برازش‌شده‌ی OLS کمتر از ۱ و ۸ مقدار صفر یا منفی‌اند، در این محاسبه از `max(ŷᵢ, 1)` به‌عنوان کف استفاده شده است:

\[
w_i=\frac{1}{\max(\hat y_i,1)^2}.
\]

این کف یک قاعده‌ی عملی برای جلوگیری از وزن‌های بسیار بزرگ در نزدیکی صفر است؛ وزن معکوسِ مربع ذاتاً منفی نمی‌شود. این انتخاب piecewise و تا حدی قراردادی است، پس باید به‌عنوان محدودیت گزارش شود و حساسیت نتایج به کف/تابع واریانس بررسی شود.

پس از این وزن‌دهی، BP `p≈0.54` می‌دهد؛ یعنی در این آزمون شواهد کافی برای رد همسانیِ باقیمانده‌های وزن‌خورده نداریم، نه این‌که همسانی واریانس اثبات شده باشد. انتخاب تابع واریانس با همین آزمونِ روی همان داده اکتشافی است.

</div>

In [47]:
ols = sm.OLS(y, X_q2).fit()
ols_hc3 = ols.get_robustcov_results('HC3')
wls = sm.WLS(y, X_q2, weights=w_fit2).fit()
huber = sm.RLM(y, X_q2, M=sm.robust.norms.HuberT()).fit()
tukey = sm.RLM(y, X_q2, M=sm.robust.norms.TukeyBiweight()).fit()

names = m_q2.model.exog_names
coefs = pd.DataFrame({'OLS': ols.params, 'WLS': wls.params, 'Huber': huber.params, 'Tukey': tukey.params}, index=names).round(4)
ses = pd.DataFrame({'OLS': ols.bse, 'OLS (HC3)': ols_hc3.bse, 'WLS': wls.bse, 'Huber': huber.bse, 'Tukey': tukey.bse}, index=names).round(4)

print('coefficients (original scale of monthly_spend):')
print(coefs)
print()
print('standard errors:')
print(ses)

coefficients (original scale of monthly_spend):
                       OLS     WLS   Huber   Tukey
Intercept           4.6674  5.2033  3.6059  3.6504
C(region)[T.Metro] -5.5449 -3.3663 -3.6926 -2.8726
C(region)[T.Other] -6.6663 -4.0958 -4.2526 -3.4244
age                -0.0731 -0.0250 -0.0377 -0.0251
income              0.1494  0.0770  0.1149  0.0828
tenure_months       0.0256  0.0107  0.0171  0.0131
n_products          0.5921  0.3352  0.4513  0.4033
investments         0.0051 -0.0043 -0.0011 -0.0016

standard errors:
                       OLS  OLS (HC3)     WLS   Huber   Tukey
Intercept           1.3619     2.0426  0.5514  0.5690  0.4690
C(region)[T.Metro]  0.6016     0.5633  0.3996  0.2513  0.2072
C(region)[T.Other]  0.5963     0.7116  0.3867  0.2491  0.2053
age                 0.0299     0.0228  0.0076  0.0125  0.0103
income              0.0127     0.0476  0.0084  0.0053  0.0044
tenure_months       0.0127     0.0092  0.0036  0.0053  0.0044
n_products          0.1588     0.1938  0.

<div dir="rtl" style="text-align: right;">

### مقایسه‌ی برآوردگرها (مقیاس اصلی)

| ضریب `income` | مقدار | SE |
|---|---|---|
| OLS | ۰٫۱۴۹۴ | ۰٫۰۱۲۷ |
| OLS با HC3 | ۰٫۱۴۹۴ | ۰٫۰۴۷۶ |
| WLS | ۰٫۰۷۷۰ | ۰٫۰۰۸۴ |
| Huber | ۰٫۱۱۴۹ | ۰٫۰۰۵۳ |
| Tukey | ۰٫۰۸۲۸ | ۰٫۰۰۴۴ |

- **SE ساده‌ی OLS دست‌کم گرفته شده است.** خطای HC3 حدود ۳٫۷ برابر SE معمولی است (۰٫۰۴۷۶ در برابر ۰٫۰۱۲۷). یعنی ناهمسانی واریانس باعث می‌شود OLS معمولی بیش از حد مطمئن به نظر برسد. ضریب OLS خودش اریب نیست، ولی SE و آزمون‌هایش قابل‌اعتماد نیستند. با HC3 هنوز `income` معنادار است (t حدود ۳٫۱).
- **ضریب `income` با روش‌های مقاوم کوچک‌تر می‌شود:** از ۰٫۱۴۹ در OLS به ۰٫۰۷۷ در WLS و ۰٫۰۸۳ در Tukey و ۰٫۱۱۵ در Huber. چند مشاهده‌ی دور (مثل مشتری‌های پردرآمد با خرج کم) شیب OLS را بالا می‌کشیدند. Huber وزن‌ها را فقط کم می‌کند ولی صفر نمی‌کند، پس بین OLS و Tukey قرار می‌گیرد.
- **ضرایب `region` هم کوچک‌تر می‌شوند** (مثلاً Other در OLS ‎−۶٫۶۷ و در Tukey ‎−۳٫۴۲). یعنی بخشی از «تفاوت منطقه‌ها» در OLS ناشی از چند تهرانی پرخرج بود.
- **`investments`** در OLS نزدیک صفر (۰٫۰۰۵۱) و با WLS ‎−۰٫۰۰۴۳ است و در همه‌ی روش‌ها اثر عملی ندارد.
- SE های Huber و Tukey از مدل مقاوم خود نرم‌افزار می‌آیند و مستقیماً با HC3 قابل‌مقایسه نیستند. SE معمولی WLS نیز وزن‌ها را ثابت و معلوم فرض می‌کند، درحالی‌که این وزن‌ها از همین داده برآورد شده‌اند؛ برای استنباط دقیق‌تر، کوواریانس مقاوم/بوت‌استرپ را بررسی کنید.
- این اختلاف ضرایب associationهای حساس به وزن‌دهی/نقاط دور را نشان می‌دهد؛ به‌تنهایی مشخص نمی‌کند کدام برآورد، اثر علّی است.

</div>

In [48]:
tw = pd.DataFrame({'id': df['id'], 'region': df['region'], 'income': df['income'],
                   'monthly_spend': df['monthly_spend'], 'ols_resid': ols.resid.round(1),
                   'tukey_weight': tukey.weights.round(3)})

print('weight == 0:', int((tukey.weights == 0).sum()), ' weight < 0.5:', int((tukey.weights < 0.5).sum()))
print()
print('mean Tukey weight by region:')
print(tw.groupby('region', observed=True)['tukey_weight'].mean().round(3).to_frame())
tw.sort_values('tukey_weight').head(12)

weight == 0: 25  weight < 0.5: 41

mean Tukey weight by region:
        tukey_weight
region              
Tehran         0.723
Metro          0.940
Other          0.905


,id,region,income,monthly_spend,ols_resid,tukey_weight
397,398,Tehran,58.9,40.72,23.7,0.0
55,56,Tehran,61.5,21.25,7.6,0.0
69,70,Tehran,36.6,15.95,5.5,0.0
65,66,Tehran,52.9,17.36,2.4,0.0
99,100,Tehran,97.4,55.82,36.2,0.0
323,324,Tehran,59.8,25.33,10.7,0.0
310,311,Other,190.0,3.50,-22.9,0.0
355,356,Tehran,25.1,45.00,35.7,0.0
74,75,Tehran,62.8,19.00,6.5,0.0
263,264,Tehran,46.3,20.74,7.9,0.0


<div dir="rtl" style="text-align: right;">

### وزن‌های Tukey

۲۵ مشاهده وزن دقیقاً صفر می‌گیرند و ۴۱ مشاهده وزن کمتر از ۰٫۵. وزن Tukey وقتی صفر می‌شود که باقی‌مانده (نسبت به مقیاس مقاوم، حدود ۱٫۵۶) از آستانه‌ی ۴٫۶۸۵ برابر آن بیشتر باشد. این‌ها مشاهده‌های **پرت** از نظر باقی‌مانده‌اند، نه لزوماً اهرمی:
- بیشترشان تهرانی‌های با خرج خیلی بالا هستند (مثل `id` 398 و 356 و 186 با باقی‌مانده‌ی مثبت ۲۴ تا ۳۶).
- چند مشتری Other با درآمد بسیار بالا و خرج کم (`id` 152 و 311 با باقی‌مانده‌ی ‎−۱۹ و ‎−۲۳).

میانگین وزن Tukey در تهران ۰٫۷۲ است، در Other ۰٫۹۱ و در Metro ۰٫۹۴. یعنی تهران بیشترین مشاهده‌ی دور را دارد، که با ناهمسانی واریانس بخش الف جور است.

**اختلاف با Cook:** Cook و اهرم نقطه‌هایی را پیدا می‌کنند که برازش را جابه‌جا می‌کنند. Tukey نقطه‌هایی را که با الگوی اکثریت جور نیستند وزن صفر می‌دهد. این دو فهرست یکی نیستند (مثلاً `id` 32 اهرم بالا دارد و وزن Tukey صفر است، ولی بسیاری از وزن‌های صفر اهرم کمی دارند).

</div>

<div dir="rtl" style="text-align: right;">

### پیش‌بینی برای مشتری جدید (مدل لگاریتمی)

مشخصات: `age = 40`، `income = 35`، `tenure_months = 60`، `n_products = 3`، `investments = 40`، `region = Tehran`.

</div>

In [49]:
x0 = pd.DataFrame({
    'age': [40], 'income': [35], 'tenure_months': [60], 'n_products': [3], 'investments': [40],
    'region': pd.Categorical(['Tehran'], categories=['Tehran', 'Metro', 'Other']),
})

pred = m_log.get_prediction(x0).summary_frame(alpha=0.05)
print('Log-response prediction:')
print(pred.round(4).to_string(index=False))

# Exponentiating the log-scale interval gives a median CI and a prediction interval,
# not a confidence interval for the arithmetic mean of Y.
back = np.exp(pred[['mean', 'mean_ci_lower', 'mean_ci_upper', 'obs_ci_lower', 'obs_ci_upper']])
print('Exponentiated: median interval and individual prediction interval (million toman):')
print(back.round(2).to_string(index=False))

eta = float(pred['mean'].iloc[0])
eta_se = float(pred['mean_se'].iloc[0])
sigma2 = float(m_log.mse_resid)
df_resid = float(m_log.df_resid)
median_hat = np.exp(eta)
mean_hat_normal = np.exp(eta + sigma2 / 2)

# Approximate delta-method CI for E[Y|x] under homoskedastic normal log errors.
# Var(log(mean_hat)) ~= Var(eta_hat) + sigma^4/(2*df_resid).
se_log_mean = np.sqrt(eta_se**2 + sigma2**2 / (2 * df_resid))
zcrit = stats.norm.ppf(0.975)
mean_ci_normal = np.exp(np.log(mean_hat_normal) + np.array([-1, 1]) * zcrit * se_log_mean)

exp_resid = pd.Series(np.exp(m_log.resid.to_numpy()), index=df.index)
smear_global = exp_resid.mean()
smear_by_region = exp_resid.groupby(df['region'], observed=True).mean()
smear_tehran = float(smear_by_region.loc['Tehran'])

# Match the formula model's column order (Tehran is the reference region).
x0_values = {
    'Intercept': 1.0, 'age': 40.0, 'income': 35.0, 'tenure_months': 60.0,
    'n_products': 3.0, 'investments': 40.0,
    'C(region)[T.Metro]': 0.0, 'C(region)[T.Other]': 0.0,
}
xv = np.array([x0_values[name] for name in m_log.model.exog_names])
h0 = xv @ np.linalg.inv(m_log.model.exog.T @ m_log.model.exog) @ xv

print(f'median estimate = {median_hat:.2f}')
print(f'normal-log mean estimate = {mean_hat_normal:.2f}; approximate 95% delta CI = [{mean_ci_normal[0]:.2f}, {mean_ci_normal[1]:.2f}]')
print(f'global Duan mean estimate = {median_hat * smear_global:.2f}')
print(f'Tehran-specific Duan estimate = {median_hat * smear_tehran:.2f} (smearing factor {smear_tehran:.4f})')
print(f'leverage h0 = {h0:.4f}')

Log-response prediction:
  mean  mean_se  mean_ci_lower  mean_ci_upper  obs_ci_lower  obs_ci_upper
2.1333   0.0432         2.0484         2.2182        1.3385        2.9281
Exponentiated: median interval and individual prediction interval (million toman):
 mean  mean_ci_lower  mean_ci_upper  obs_ci_lower  obs_ci_upper
 8.44           7.76           9.19          3.81         18.69
median estimate = 8.44
normal-log mean estimate = 9.15; approximate 95% delta CI = [8.40, 9.97]
global Duan mean estimate = 9.13
Tehran-specific Duan estimate = 9.32 (smearing factor 1.1036)
leverage h0 = 0.0115


<div dir="rtl" style="text-align: right;">

برای مشتری خواسته‌شده، روی مقیاس `log(y)`، `η̂=۲٫۱۳۳۳` و SE برازش ۰٫۰۴۳۲ است. با تبدیل معکوس، بازه‌ی ۷٫۷۶ تا ۹٫۱۹ **بازه‌ی اطمینان میانه‌ی شرطی** است؛ فاصله‌ی پیش‌بینیِ مشاهده‌ی جدید ۳٫۸۱ تا ۱۸٫۶۹ میلیون تومان است.

برای میانگین حسابی پاسخ، تصحیح نرمال `exp(σ̂²/2)` برآورد نقطه‌ای حدود ۹٫۱۵ و فاصله‌ی اطمینان دلتا‌متدی تقریبی حدود ۸٫۴۰ تا ۹٫۹۷ میلیون تومان می‌دهد، تحت فرض خطاهای لگاریتمی نرمال و همسان‌واریانس. چون آزمون BP در مقیاس log هنوز ناهمسانی را رد می‌کند، این فاصله و فاصله‌ی پیش‌بینی مدل‌محور را باید با احتیاط تفسیر کرد؛ بازه‌ی ۷٫۷۶ تا ۹٫۱۹ را به‌اشتباه CI میانگین حسابی ننامید.

Duan با smearing سراسری حدود ۹٫۱۳ می‌دهد؛ چون پراکندگی با منطقه فرق دارد، کد یک ضریب Tehran-specific را هم برای این مشتری گزارش می‌کند. این ضریب منطقه‌ای ناهمسانی بین منطقه‌ها را تا حدی لحاظ می‌کند، اما فرض می‌کند توزیع خطا درون منطقه نیز با سایر ویژگی‌ها تغییر مهمی نکند. اهرم مشتری حدود ۰٫۰۱۱۵ و زیر آستانه‌ی ۰٫۰۴ است.

</div>

<div dir="rtl" style="text-align: right;">

## سوال ۲-د (مفهومی)

### i) OLS تحت ناهمسانی واریانس

اگر `E(ε|X)=0` و ماتریس طرح رتبه‌ی کامل داشته باشد، ناهمسانی واریانس به‌تنهایی نااریبی شرطی OLS را از بین نمی‌برد. سازگاری نیز تحت شرایط منظم نمونه‌گیری (مانند قانون اعداد بزرگ و لحظات متناهی) برقرار است؛ «فقط میانگین صفر» به‌تنهایی برای اثبات سازگاری کافی نیست.

ناهمسانی، فرمول معمول کوواریانسِ همسان‌واریانس و بنابراین SE/t/p معمول را نامعتبر می‌کند و OLS دیگر در کلاس برآوردگرهای خطی نااریب کاراترین نیست. در خروجی این داده، SE درآمد با HC3 از ۰٫۰۱۲۷ به ۰٫۰۴۷۶ می‌رود.

### ii) ساندویچ OLS و شرط BLUE بودن WLS

با `A=(X'X)⁻¹` و `β̂−β=A X'ε`، به شرط ثابت بودن `X` داریم:

\[
\operatorname{Var}(\hat\beta\mid X)
=A X'\operatorname{Var}(\epsilon\mid X)X A'
=(X'X)^{-1}X'\Sigma X(X'X)^{-1}.
\]

برای HC0، `Σ` را با ماتریس قطریِ مربع باقیمانده‌ها برآورد می‌کنند؛ HC3 اصلاح اهرم را نیز اعمال می‌کند.

WLS وقتی BLUE است که وزن‌ها متناسب با `Σ⁻¹` باشند و فرض‌های خطی گاوس–مارکف برقرار باشند. در حالت خطاهای ناهمبسته، `W=diag(1/σ_i²)`؛ وزن‌کردن پاسخ و طراحی با `√W` خطاهای تبدیل‌شده را هم‌واریانس می‌کند. اگر وزن‌های معلوم/ثابت غلط باشند، با `E(ε|X)=0` برآورد هنوز نااریب است ولی کارا نیست؛ وقتی وزن‌ها از همان داده برآورد می‌شوند، استنباط معمول WLS عدم‌قطعیت مرحله‌ی برآورد وزن را نادیده می‌گیرد و BLUE بودن دقیق نیست.

### iii) مدل لگاریتمی: میانه و میانگین

در `log(Y)=Xβ+ε`، چون `exp` صعودی است،

\[
\operatorname{Median}(Y\mid X=x)
=\exp\{x'\beta+\operatorname{Median}(\epsilon\mid X=x)\}.
\]

پس `exp(x'β̂)` برآورد میانه است **اگر** میانه‌ی شرطی خطا صفر باشد (برای مثال، خطای شرطی متقارن حول صفر). برای میانگین حسابی:

\[
E(Y\mid X=x)=\exp(x'\beta)\,E\{\exp(\epsilon)\mid X=x\}.
\]

اگر `ε|X=x ~ N(0,σ²)`، آنگاه `E(Y|x)=exp(x'β+σ²/2)`؛ این همان تصحیح نرمال است. اگر واریانس با `x` تغییر کند، باید از `σ²(x)` استفاده کرد، نه یک واریانس سراسری.

تصحیح Duan، `exp(x'β̂) × mean(exp(residual))`، نرمال‌بودن را لازم ندارد. اما یک smearing factor سراسری مستلزم آن است که `E(exp(ε)|X=x)` در نقاط پیش‌بینی تقریباً ثابت باشد (معمولاً با یک توزیع شرطی خطای ثابت در سراسر X توجیه می‌شود). ثابت بودن واریانس به‌تنهایی تضمین کافی نیست. با تفاوت SD باقی‌مانده‌ی log بین منطقه‌ها (Tehran ۰٫۴۱۷، Metro ۰٫۲۸۰، Other ۰٫۴۶۸)، ضریب سراسری تقریبی است؛ smearing منطقه‌ای/شرطی می‌تواند مناسب‌تر باشد.

برای مشتری جدید، `exp(η̂)=۸٫۴۴` برآورد میانه است؛ تصحیح نرمال سراسری حدود ۹٫۱۵، Duan سراسری حدود ۹٫۱۳، و Duan ویژه‌ی تهران حدود ۹٫۳۲ می‌دهد. در کل نمونه، میانگین پیش‌بینی خام حدود ۵٫۸۵ بود؛ تصحیح نرمال و Duan سراسری آن را به‌ترتیب به ۶٫۳۴ و ۶٫۳۳ رساند، درحالی‌که میانگین پاسخ ۶٫۵۶ است. این کالیبراسیون روی همان داده، تضمین عملکرد خارج از نمونه نیست.

</div>